# MPNN → Boltz: Sequence Design & Structure Prediction Pipeline

This notebook:
1. Parses LigandMPNN output `.fa` files to extract designed sequences and metrics
2. Generates per-sequence YAML input files for Boltz structure prediction
3. Runs Boltz prediction (via shell command)
4. Parses Boltz confidence outputs and merges with MPNN metrics
5. Saves a combined DataFrame with all metrics, identifiers, and file paths

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

In [30]:
# === Configuration ===
MPNN_SEQS_DIR = Path("../data/sequences/mpnn_toxin/seqs")
MPNN_BACKBONES_DIR = Path("../data/sequences/mpnn_toxin/backbones")
BOLTZ_INPUT_DIR = Path("../data/sequences/mpnn_toxin/boltz_input_re")
BOLTZ_OUTPUT_DIR = Path("../data/sequences/mpnn_toxin/boltz_output_re")
OUTPUT_PKL = Path("../data/sequences/mpnn_toxin/mpnn_boltz_metrics_re.pkl")
FILTERED_OUTPUT_PKL = Path("../data/sequences/mpnn_toxin/mpnn_boltz_filtered_metrics_re.pkl")

BOLTZ_INPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Parse MPNN Output Files

In [9]:
def parse_mpnn_fa(fa_path: Path) -> list[dict]:
    """Parse a single MPNN .fa file into a list of records.

    Each .fa file contains:
    - Line 1: header for native sequence (model info)
    - Line 2: native sequence
    - Lines 3+: pairs of (header, sequence) for each designed variant
      with metrics: id, T, seed, overall_confidence, ligand_confidence, seq_rec
    """
    lines = fa_path.read_text().strip().split("\n")
    model_name = fa_path.stem  # e.g. AF-P05516-F1-model_v6

    # Extract parent UniProt ID from filename: AF-{UNIPROT_ID}-F1-model_v6
    match = re.match(r"AF-(.+?)-F1-model_", model_name)
    parent_id = match.group(1) if match else model_name

    # Line 0: native header, Line 1: native sequence
    native_seq = lines[1]

    records = []
    # Designed sequences start at line 2, in pairs (header, seq)
    for i in range(2, len(lines), 2):
        header = lines[i]
        seq = lines[i + 1] if i + 1 < len(lines) else ""

        # Parse header: >name, id=N, T=X, seed=X, overall_confidence=X, ...
        fields = {}
        for part in header.lstrip(">").split(", "):
            if "=" in part:
                k, v = part.split("=", 1)
                fields[k.strip()] = v.strip()

        design_id = int(fields.get("id", 0))
        records.append({
            "parent_id": parent_id,
            "model_name": model_name,
            "design_id": design_id,
            "native_sequence": native_seq,
            "designed_sequence": seq,
            "seq_len": len(seq),
            "mpnn_temperature": float(fields.get("T", 0)),
            "mpnn_seed": int(fields.get("seed", 0)),
            "mpnn_overall_confidence": float(fields.get("overall_confidence", 0)),
            "mpnn_ligand_confidence": float(fields.get("ligand_confidence", 0)),
            "mpnn_seq_rec": float(fields.get("seq_rec", 0)),
            "mpnn_fa_path": str(fa_path),
            "mpnn_backbone_path": str(MPNN_BACKBONES_DIR / f"{model_name}_{design_id}.pdb"),
        })

    return records

In [10]:
fa_files = sorted(MPNN_SEQS_DIR.glob("*.fa"))
print(f"Found {len(fa_files)} MPNN output files")

all_records = []
for fa in fa_files:
    all_records.extend(parse_mpnn_fa(fa))

df = pd.DataFrame(all_records)
print(f"Total designed sequences: {len(df)}")
print(f"Unique parent proteins: {df['parent_id'].nunique()}")
df.head()

Found 6813 MPNN output files
Total designed sequences: 136260
Unique parent proteins: 6813


,parent_id,model_name,design_id,native_sequence,designed_sequence,seq_len,mpnn_temperature,mpnn_seed,mpnn_overall_confidence,mpnn_ligand_confidence,mpnn_seq_rec,mpnn_fa_path,mpnn_backbone_path
0,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,1,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,VPILDVADIPPEVFADPALGAVAAAALASGAGLL,34,0.1,111,0.2922,1.0,0.2353,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...
1,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,2,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,VPVIDVEDIPPEVFEDPELGEPVRALLESGEGLD,34,0.1,111,0.3015,1.0,0.3235,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...
2,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,3,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,RVIIDPEDIPPEIFADPALGEVARALLESGEGLL,34,0.1,111,0.2864,1.0,0.3529,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...
3,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,4,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,VEVIDVDDIPPEIFEDPELGKKVREVLASGKGLL,34,0.1,111,0.2754,1.0,0.2647,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...
4,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,5,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,EVEIDVEDIPKEIFEDPELGKEVKEILDSGEGLL,34,0.1,111,0.2973,1.0,0.2941,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...


In [11]:
df[["mpnn_overall_confidence", "mpnn_ligand_confidence", "mpnn_seq_rec", "seq_len"]].describe()

,mpnn_overall_confidence,mpnn_ligand_confidence,mpnn_seq_rec,seq_len
count,136260.000000,136260.0,136260.000000,136260.000000
mean,0.374204,1.0,0.354309,148.585645
std,0.075276,0.0,0.126072,214.244777
min,0.142700,1.0,0.000000,16.000000
25%,0.313500,1.0,0.250000,61.000000
50%,0.376100,1.0,0.363000,83.000000
75%,0.435000,1.0,0.456500,138.000000
max,0.594600,1.0,0.743600,2367.000000


## 2. Pre-filter: Discard Sequences with 5+ Consecutive Alanines

Poly-alanine stretches are a known MPNN failure mode — they indicate the model defaulted to a low-complexity region. Discard any designed sequence containing 5 or more consecutive alanine residues.

In [12]:
MIN_CONSECUTIVE_ALA = 5
poly_ala_pattern = re.compile(r"A{" + str(MIN_CONSECUTIVE_ALA) + r",}")

has_poly_ala = df["designed_sequence"].str.contains(poly_ala_pattern)
n_filtered = has_poly_ala.sum()

print(f"Sequences with {MIN_CONSECUTIVE_ALA}+ consecutive alanines: {n_filtered}/{len(df)} ({n_filtered/len(df)*100:.1f}%)")
df = df[~has_poly_ala].reset_index(drop=True)
print(f"Remaining sequences after filtering: {len(df)}")
print(f"Unique parent proteins remaining: {df['parent_id'].nunique()}")

Sequences with 5+ consecutive alanines: 55612/136260 (40.8%)
Remaining sequences after filtering: 80648
Unique parent proteins remaining: 6328


## 3. Generate Boltz Input YAML Files

Each designed sequence gets its own YAML file for Boltz prediction.  
Since these are *designed* (non-natural) sequences, MSA is set to `empty` (single-sequence mode).

In [13]:
def make_boltz_yaml(row: pd.Series, output_dir: Path) -> str:
    """Create a Boltz input YAML for a single designed sequence."""
    name = f"{row['model_name']}_design_{row['design_id']}"
    yaml_path = output_dir / f"{name}.yaml"
    yaml_content = (
        f"version: 1\n"
        f"sequences:\n"
        f"  - protein:\n"
        f"      id: A\n"
        f"      sequence: {row['designed_sequence']}\n"
        f"      msa: empty\n"
    )
    yaml_path.write_text(yaml_content)
    return str(yaml_path)

In [14]:
df["boltz_input_path"] = df.apply(make_boltz_yaml, axis=1, output_dir=BOLTZ_INPUT_DIR)
print(f"Created {len(df)} Boltz input YAML files in {BOLTZ_INPUT_DIR}")

# Preview one
print("\n--- Example YAML ---")
print(Path(df["boltz_input_path"].iloc[0]).read_text())

Created 80648 Boltz input YAML files in ../data/sequences/mpnn_toxin/boltz_input_re

--- Example YAML ---
version: 1
sequences:
  - protein:
      id: A
      sequence: VPILDVADIPPEVFADPALGAVAAAALASGAGLL
      msa: empty



## 4. Run Boltz Structure Prediction

Run from the **project root** with the `boltz` pixi environment:
```bash
pixi run -e boltz boltz predict data/sequences/mpnn_toxin/boltz_input \
    --out_dir data/sequences/mpnn_toxin/boltz_output \
    --recycling_steps 3 \
    --sampling_steps 200 \
    --diffusion_samples 1 \
    --output_format pdb \
    --override
```

Uncomment the cell below to run from the notebook, or run the command in a terminal.

In [ ]:
# import subprocess
# cmd = [
#     "pixi", "run", "-e", "boltz",
#     "boltz", "predict", str(BOLTZ_INPUT_DIR),
#     "--out_dir", str(BOLTZ_OUTPUT_DIR),
#     "--recycling_steps", "3",
#     "--sampling_steps", "200",
#     "--diffusion_samples", "1",
#     "--output_format", "pdb",
#     "--override",
# ]
# result = subprocess.run(cmd, cwd="..", capture_output=True, text=True)
# print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr[-2000:])

## 5. Parse Boltz Outputs & Merge Metrics

In [15]:
def find_boltz_predictions(boltz_out_dir: Path) -> Path | None:
    """Find the predictions/ directory inside boltz_results_*."""
    results_dirs = sorted(boltz_out_dir.glob("boltz_results_*/predictions"))
    if results_dirs:
        return results_dirs[0]
    return None


def parse_boltz_confidence(pred_dir: Path, name: str) -> dict | None:
    """Parse Boltz confidence JSON for a given prediction name."""
    sample_dir = pred_dir / name
    if not sample_dir.exists():
        return None

    # Find confidence JSON (model_0 is the top-ranked)
    conf_files = sorted(sample_dir.glob("confidence_*_model_0.json"))
    if not conf_files:
        return None

    with open(conf_files[0]) as f:
        conf = json.load(f)

    # Find structure file
    struct_files = sorted(sample_dir.glob(f"{name}_model_0.*"))
    struct_path = str(struct_files[0]) if struct_files else ""

    return {
        "boltz_confidence_score": conf.get("confidence_score"),
        "boltz_ptm": conf.get("ptm"),
        "boltz_iptm": conf.get("iptm"),
        "boltz_plddt": conf.get("complex_plddt"),
        "boltz_iplddt": conf.get("complex_iplddt"),
        "boltz_pde": conf.get("complex_pde"),
        "boltz_ipde": conf.get("complex_ipde"),
        "boltz_structure_path": struct_path,
        "boltz_confidence_path": str(conf_files[0]),
    }

In [16]:
pred_dir = find_boltz_predictions(BOLTZ_OUTPUT_DIR)

if pred_dir is None:
    print(f"No Boltz predictions found in {BOLTZ_OUTPUT_DIR}.")
    print("Run Boltz first (see cell above), then re-run this cell.")
else:
    print(f"Boltz predictions directory: {pred_dir}")
    available = sorted(pred_dir.iterdir())
    print(f"Found {len(available)} prediction directories")

    # Build boltz name from df columns
    df["boltz_name"] = df["model_name"] + "_design_" + df["design_id"].astype(str)

    boltz_records = []
    for _, row in df.iterrows():
        metrics = parse_boltz_confidence(pred_dir, row["boltz_name"])
        boltz_records.append(metrics or {})

    df_boltz = pd.DataFrame(boltz_records, index=df.index)
    df = pd.concat([df, df_boltz], axis=1)

    n_with_boltz = df["boltz_confidence_score"].notna().sum()
    print(f"Matched {n_with_boltz}/{len(df)} sequences with Boltz predictions")

Boltz predictions directory: ../data/sequences/mpnn_toxin/boltz_output_re/boltz_results_boltz_input_re/predictions
Found 65831 prediction directories
Matched 65831/80648 sequences with Boltz predictions


In [17]:
# Summary statistics for Boltz metrics
boltz_cols = [c for c in df.columns if c.startswith("boltz_") and df[c].dtype != object]
if boltz_cols:
    display(df[boltz_cols].describe())
else:
    print("No Boltz metrics available yet.")

,boltz_confidence_score,boltz_ptm,boltz_iptm,boltz_plddt,boltz_iplddt,boltz_pde,boltz_ipde
count,65831.000000,65831.000000,65831.0,65831.000000,65831.000000,65831.000000,65831.0
mean,0.700920,0.569519,0.0,0.733771,0.733771,1.293228,0.0
std,0.160431,0.213715,0.0,0.162828,0.162828,1.030749,0.0
min,0.263124,0.054296,0.0,0.269492,0.269492,0.261033,0.0
25%,0.589150,0.398912,0.0,0.620120,0.620120,0.537464,0.0
50%,0.729128,0.561066,0.0,0.766533,0.766533,0.934775,0.0
75%,0.826493,0.760637,0.0,0.864850,0.864850,1.666959,0.0
max,0.965914,0.960881,0.0,0.989176,0.989176,6.180642,0.0


## 6. Filtering & Exploration

Filter designed sequences by quality thresholds.

In [18]:
# Configurable quality thresholds
PLDDT_THRESHOLD = 0.7
PTM_THRESHOLD = 0.5
MPNN_CONFIDENCE_THRESHOLD = 0.3

if "boltz_plddt" in df.columns and df["boltz_plddt"].notna().any():
    df_good = df[
        (df["boltz_plddt"] >= PLDDT_THRESHOLD)
        & (df["boltz_ptm"] >= PTM_THRESHOLD)
        & (df["mpnn_overall_confidence"] >= MPNN_CONFIDENCE_THRESHOLD)
    ].copy()
    print(
        f"High-quality designs: {len(df_good)}/{len(df)} "
        f"({len(df_good)/len(df)*100:.1f}%)"
    )
    print(f"Unique parent proteins with good designs: {df_good['parent_id'].nunique()}")
    display(df_good.head())
else:
    print("Run Boltz first to enable quality filtering.")

High-quality designs: 27798/80648 (34.5%)
Unique parent proteins with good designs: 3202


,parent_id,model_name,design_id,native_sequence,designed_sequence,seq_len,mpnn_temperature,mpnn_seed,mpnn_overall_confidence,mpnn_ligand_confidence,...,boltz_name,boltz_confidence_score,boltz_ptm,boltz_iptm,boltz_plddt,boltz_iplddt,boltz_pde,boltz_ipde,boltz_structure_path,boltz_confidence_path
1,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,2,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,VPVIDVEDIPPEVFEDPELGEPVRALLESGEGLD,34,0.1,111,0.3015,1.0,...,AF-A0A023IWD9-F1-model_v6_design_2,0.798245,0.634326,0.0,0.839224,0.839224,0.504492,0.0,../data/sequences/mpnn_toxin/boltz_output_re/b...,../data/sequences/mpnn_toxin/boltz_output_re/b...
17,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,18,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,VPVLDPADIPPSVFEDPALGAPARAALASGAGLA,34,0.1,111,0.3089,1.0,...,AF-A0A023IWD9-F1-model_v6_design_18,0.781932,0.590373,0.0,0.829822,0.829822,0.812229,0.0,../data/sequences/mpnn_toxin/boltz_output_re/b...,../data/sequences/mpnn_toxin/boltz_output_re/b...
19,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,20,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,VPVVDVADIPPEVFADPALGAPLRAALDSGAGLL,34,0.1,111,0.3146,1.0,...,AF-A0A023IWD9-F1-model_v6_design_20,0.774729,0.612311,0.0,0.815334,0.815334,0.528199,0.0,../data/sequences/mpnn_toxin/boltz_output_re/b...,../data/sequences/mpnn_toxin/boltz_output_re/b...
62,A0A023IWE2,AF-A0A023IWE2-F1-model_v6,4,MSDINATRLPIWGIGCDPCVGDDVTAVLTRGEA,ALDLAALLLPDPAAGGLPPAALAALAAALALLR,33,0.1,111,0.3211,1.0,...,AF-A0A023IWE2-F1-model_v6_design_4,0.735688,0.551863,0.0,0.781645,0.781645,0.814145,0.0,../data/sequences/mpnn_toxin/boltz_output_re/b...,../data/sequences/mpnn_toxin/boltz_output_re/b...
64,A0A023IWE2,AF-A0A023IWE2-F1-model_v6,9,MSDINATRLPIWGIGCDPCVGDDVTAVLTRGEA,PLPLGLLLLPLPGLGGLPPAALAALAALLALLR,33,0.1,111,0.3218,1.0,...,AF-A0A023IWE2-F1-model_v6_design_9,0.800537,0.508101,0.0,0.873646,0.873646,0.959078,0.0,../data/sequences/mpnn_toxin/boltz_output_re/b...,../data/sequences/mpnn_toxin/boltz_output_re/b...


## 7. Save Combined DataFrame

In [19]:
df.to_pickle(OUTPUT_PKL)
print(f"Saved {len(df)} records to {OUTPUT_PKL}")
print(f"\nColumns: {list(df.columns)}")
print(f"DataFrame shape: {df.shape}")

Saved 80648 records to ../data/sequences/mpnn_toxin/mpnn_boltz_metrics_re.pkl

Columns: ['parent_id', 'model_name', 'design_id', 'native_sequence', 'designed_sequence', 'seq_len', 'mpnn_temperature', 'mpnn_seed', 'mpnn_overall_confidence', 'mpnn_ligand_confidence', 'mpnn_seq_rec', 'mpnn_fa_path', 'mpnn_backbone_path', 'boltz_input_path', 'boltz_name', 'boltz_confidence_score', 'boltz_ptm', 'boltz_iptm', 'boltz_plddt', 'boltz_iplddt', 'boltz_pde', 'boltz_ipde', 'boltz_structure_path', 'boltz_confidence_path']
DataFrame shape: (80648, 24)


In [31]:
df_good.to_pickle(FILTERED_OUTPUT_PKL)
print(f"Saved {len(df_good)} records to {FILTERED_OUTPUT_PKL}")
print(f"\nColumns: {list(df_good.columns)}")
print(f"DataFrame shape: {df_good.shape}")

Saved 27798 records to ../data/sequences/mpnn_toxin/mpnn_boltz_metrics_re.pkl

Columns: ['parent_id', 'model_name', 'design_id', 'native_sequence', 'designed_sequence', 'seq_len', 'mpnn_temperature', 'mpnn_seed', 'mpnn_overall_confidence', 'mpnn_ligand_confidence', 'mpnn_seq_rec', 'mpnn_fa_path', 'mpnn_backbone_path', 'boltz_input_path', 'boltz_name', 'boltz_confidence_score', 'boltz_ptm', 'boltz_iptm', 'boltz_plddt', 'boltz_iplddt', 'boltz_pde', 'boltz_ipde', 'boltz_structure_path', 'boltz_confidence_path']
DataFrame shape: (27798, 24)


## 8. Generate Test Sets for ProteNote Models

Creates test data for two model variants:
- **Sequence-only model**: FASTA file with `>seq_id GO:term1 GO:term2 ...` headers, where GO terms are inherited from the parent protein
- **Hybrid model (ESM-C + EGNN)**: Same FASTA + `structure_index.json` pointing to Boltz-predicted CIF structures. Then run `bin/generate_sequence_embeddings.py` and `bin/prepare_graph_data.py` to produce the ESM-C embeddings and atom-level graphs.

Each designed sequence gets a unique `seq_id`: `{parent_id}_d{design_id}` (e.g., `P05516_d3`).

In [32]:
import pickle
import shutil

# Load the filtered MPNN+Boltz DataFrame
df_test = pd.read_pickle(FILTERED_OUTPUT_PKL)

# Load parent protein GO annotations
df_parents = pd.read_pickle("../data/swissprot/uniprot_sprot_2025_01_toxin.pkl")
parent_go = df_parents.set_index("seq_id")["go_ids"].to_dict()

# Assign unique seq_id per designed sequence
df_test["seq_id"] = df_test["parent_id"] + "_d" + df_test["design_id"].astype(str)

# Inherit GO terms from parent
df_test["go_ids"] = df_test["parent_id"].map(parent_go)

# Drop any rows whose parent has no GO annotations
n_before = len(df_test)
df_test = df_test[df_test["go_ids"].apply(lambda x: isinstance(x, list) and len(x) > 0)].reset_index(drop=True)
print(f"Dropped {n_before - len(df_test)} designs with no parent GO annotations")
print(f"Test set size: {len(df_test)} designs from {df_test['parent_id'].nunique()} parents")
print(f"Example seq_id: {df_test['seq_id'].iloc[0]}")
print(f"Example GO terms: {df_test['go_ids'].iloc[0]}")

Dropped 0 designs with no parent GO annotations
Test set size: 27798 designs from 3202 parents
Example seq_id: A0A023IWD9_d2
Example GO terms: ['GO:0090729']


### 8a. Sequence-only test set: FASTA file

In [33]:
FASTA_OUT = Path("../data/sequences/mpnn_toxin/test_mpnn_GO.fasta")
FASTA_OUT.parent.mkdir(parents=True, exist_ok=True)

with open(FASTA_OUT, "w") as f:
    for _, row in df_test.iterrows():
        go_str = " ".join(row["go_ids"])
        f.write(f">{row['seq_id']} {go_str}\n")
        # Write sequence in 60-char lines (standard FASTA)
        seq = row["designed_sequence"]
        for i in range(0, len(seq), 60):
            f.write(seq[i:i+60] + "\n")

print(f"Wrote {len(df_test)} sequences to {FASTA_OUT}")

# Preview
with open(FASTA_OUT) as f:
    for _ in range(6):
        print(f.readline(), end="")

Wrote 27798 sequences to ../data/sequences/mpnn_toxin/test_mpnn_GO.fasta
>A0A023IWD9_d2 GO:0090729
VPVIDVEDIPPEVFEDPELGEPVRALLESGEGLD
>A0A023IWD9_d18 GO:0090729
VPVLDPADIPPSVFEDPALGAPARAALASGAGLA
>A0A023IWD9_d20 GO:0090729
VPVVDVADIPPEVFADPALGAPLRAALDSGAGLL


### 8b. Hybrid model test set: structure_index.json + processing commands

The hybrid model additionally needs:
1. `structure_index.json` — mapping each `seq_id` to its Boltz CIF structure
2. ESM-C embeddings — generated via `bin/generate_sequence_embeddings.py`
3. Atom-level graphs — generated via `bin/prepare_graph_data.py`

This cell creates the structure index. Run the two scripts afterwards (see below).

In [34]:
STRUCT_OUT_DIR = Path("../data/sequences/mpnn_toxin/structures")
STRUCT_INDEX_OUT = Path("../data/sequences/mpnn_toxin/structure_index.json")
STRUCT_OUT_DIR.mkdir(parents=True, exist_ok=True)

structure_index = {}
n_copied = 0
n_missing = 0

for _, row in df_test.iterrows():
    seq_id = row["seq_id"]
    boltz_cif = row.get("boltz_structure_path", "")

    if not boltz_cif or not Path(boltz_cif).exists():
        n_missing += 1
        continue

    # Copy/symlink the Boltz CIF into the structures directory
    dest = STRUCT_OUT_DIR / f"{seq_id}.cif"
    if not dest.exists():
        shutil.copy2(boltz_cif, dest)
    n_copied += 1

    structure_index[seq_id] = {
        "source": "boltz",
        "path": f"structures/{seq_id}.cif",
        "chain_ids": ["A"],
    }

with open(STRUCT_INDEX_OUT, "w") as f:
    json.dump(structure_index, f, indent=2)

print(f"Copied {n_copied} Boltz CIF structures to {STRUCT_OUT_DIR}")
print(f"Missing: {n_missing}")
print(f"Structure index: {STRUCT_INDEX_OUT} ({len(structure_index)} entries)")

Copied 27798 Boltz CIF structures to ../data/sequences/mpnn_toxin/structures
Missing: 0
Structure index: ../data/sequences/mpnn_toxin/structure_index.json (27798 entries)


### 8c. Processing commands for hybrid model

After running the cells above, execute these from the **project root**:

```bash
# 1. Generate ESM-C embeddings for the designed sequences
pixi run python bin/generate_sequence_embeddings.py --paths mpnn_toxin --fasta-path-names TEST_DATA_PATH

# 2. Build atom-level graphs from Boltz structures + ESM-C embeddings
pixi run python bin/prepare_graph_data.py --paths mpnn_toxin --fasta-path-names TEST_DATA_PATH
```